In [ ]:
import numpy as np
from adaptive_latents import StreamingKalmanFilter, ArrayWithTime, Bubblewrap, sjPCA
import matplotlib.pyplot as plt
from scipy.stats import special_ortho_group
from adaptive_latents.utils import principle_angles
from copy import deepcopy

rng = np.random.default_rng(0)

In [ ]:
n_rotations = 10
noise_scale = 1e-7
theta = np.arange(0, 2*n_rotations*np.pi, .1)
y = np.column_stack((np.cos(theta), np.sin(theta)))
y = y + rng.normal(loc=0,scale=noise_scale, size=y.shape)
y = ArrayWithTime(y, np.linspace(0, n_rotations, theta.size))

axis = special_ortho_group.rvs(2)[:,:1]
y_part = ArrayWithTime(y[:,0:1] @  axis.T, y.t)

y_part = np.column_stack((np.cos(theta), np.cos(theta)))
y_part = y_part + rng.normal(loc=0,scale=noise_scale, size=y_part.shape)
y_part = ArrayWithTime(y_part, np.linspace(0, n_rotations, theta.size))



In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(y.t, y[:,0], '.')
ax.plot(y_part.t, y_part[:,0], '.')


In [ ]:
%matplotlib inline
fig, ax = plt.subplots(figsize=(10,5))

kf = StreamingKalmanFilter(steps_between_refits=25,log_level=2, check_dt=True)
kf.offline_run_on(y)
error = ArrayWithTime.from_list(kf.log['pred_error'], squeeze_type='to_2d')
ax.plot(error.t, np.linalg.norm(error,axis=1), '.')

kf = StreamingKalmanFilter(steps_between_refits=25,log_level=2, check_dt=True)
kf.offline_run_on(y_part)
error = ArrayWithTime.from_list(kf.log['pred_error'], squeeze_type='to_2d')
ax.plot(error.t, np.linalg.norm(error,axis=1), '.')

# ax.axhline(np.linalg.norm(np.diff(y_part, axis=0), axis=1).max(), color='r')
ax.plot(y_part.t[1:],np.linalg.norm(np.diff(y_part, axis=0), axis=1), '-', alpha=.5)
ax.legend(['1 step pred error, [sin, cos]', '1 step pred error, [cos, cos]', '$\cos(\\theta_t) - \cos(\\theta_{t-1})$'], framealpha=1)


In [ ]:
%matplotlib inline
fig, axs = plt.subplots(figsize=(10,10), nrows=3)

for j in range(5):
    for i in range(4):
        Q = special_ortho_group.rvs(12)[:,:2]
        y_highd = y @ Q.T + rng.normal(loc=0, scale=10**(-4), size=(y.t.size, Q.shape[0]))
        match i:
            case 0:
                pass
            case 1:
                rng.shuffle(y_highd, axis=0)
            case 2:
                rng.shuffle(y_highd, axis=1)
            case 3:
                y_highd = rng.permuted(y_highd.flatten()).reshape(y_highd.shape)

        jpca = sjPCA(log_level=2)
        jpca.offline_run_on(y_highd)


        us = ArrayWithTime.from_list(jpca.log['U'])
        # distances = [principle_angles(u[:,:2], us[-1][:,:2]).sum() for u in us]
        distances = [principle_angles(u[:,:2], Q).sum() for u in us]
        axs[0].plot(np.log(distances), '-', color=f'C{i}')

        distances = [principle_angles(u[:,:2], us[-1,:,:2]).sum() for u in us]
        axs[1].plot(np.log(distances), '-', color=f'C{i}')
        axs[1].axis(axs[0].axis())

        distances = np.linalg.norm(np.diff(us[:,:2,:], axis=0), axis=(1,2))
        axs[2].plot(np.log(distances), '-', color=f'C{i}')

axs[0].set_title('Distance to true subspace (log sum of principal angles)')
axs[1].set_title('Distance to final estimated subspace (log sum of principal angles)')
axs[2].set_title('Change in estimated subspace (log Frobenius norm of difference)')
axs[2].legend(['unchanged rotational data', 'time shuffled', 'dim shuffled', 'fully shuffled'])



In [ ]:
%matplotlib inline
from adaptive_latents import Pipeline
from adaptive_latents.stim_regressor import StimRegressor, StimAutoReg


# y_stimmed = deepcopy(y); title = 'A'
y_stimmed = deepcopy(y_part); title = 'B'

y_stimmed = ArrayWithTime(np.column_stack([y_stimmed, np.zeros(y_stimmed.shape[0])]), y_stimmed.t)

out_axis = -1
n_stims = 50
stims = ArrayWithTime(np.ones(n_stims), sorted(rng.choice(y.t[26:], size=n_stims, replace=False)))
for t in stims.t:
    idx = y_stimmed.time_to_sample(t)
    y_stimmed[idx,out_axis] += 1 * 4*y_stimmed[idx, 0] ** 3

k = np.hstack([0*np.arange(9), np.exp(-np.arange(10))])
y_stimmed[:,out_axis] = np.convolve(k,y_stimmed[:,out_axis], 'same')
y_stimmed_pre_proj = y_stimmed

Q = np.eye(3)
Q = special_ortho_group.rvs(8)[:,:3]
# y_stimmed = y_stimmed @ Q.T
y_stimmed = y_stimmed + rng.normal(loc=0,scale=1e-6, size=y_stimmed.shape)

fig, ax = plt.subplots(figsize=(10,5))

srs = {
    'blind': StimRegressor(log_level=2, heed_stimuli=False, attempt_correction=False),
    'aware': StimRegressor(log_level=2)
}
errors = {}

for sr in srs.values():
    sr.autoreg.check_dt = True
# srs['aware'].stim_reg.length_scales = np.array([6.30957344e+02, 2.81838293e-10, 1.88364909e-05])

# dt_X = ArrayWithTime(y_stimmed.dt + 0*y_stimmed.t.reshape((-1,1)), y_stimmed.t)

for key, sr in srs.items():
    # y_stimmed_reduced = sjPCA().offline_run_on(y_stimmed)[:,:3]
    y_stimmed_reduced = y_stimmed
    Pipeline([
        sr,
    ]).offline_run_on([(stims,'stim'), (y_stimmed_reduced, 'X')])
    error = errors[key] = ArrayWithTime.from_list(sr.log['pred_error'], squeeze_type='to_2d')
    ax.plot(error.t, np.linalg.norm(error,axis=1), '.', label=key)

ax.legend()
ax.set_xlabel('time')
ax.set_ylabel('1step prediction error')
for t in stims.t:
    ax.axvline(t, color='red', alpha=.1)
ax.semilogy()

# ax.plot(y.t[1:],np.abs(np.diff(y[:,0])), '-', alpha=.5)
# ax.legend(['1 step pred error, [sin, cos]', '1 step pred error, [cos, cos]', '$\cos(\\theta_t) - \cos(\\theta_{t-1})$'], framealpha=1)



In [ ]:
fig, ax = plt.subplots(figsize=(6,6), subplot_kw=dict(projection='3d'))

ax.scatter(y_stimmed_pre_proj[:,0], y_stimmed_pre_proj[:,1], y_stimmed_pre_proj[:,2], s=14)

fig.savefig('/home/jgould/Downloads/nonrotational_dynamics.svg')


In [ ]:
fig, ax = plt.subplots(figsize=(6,6))
y_stimmed_reduced = sjPCA().offline_run_on(y_stimmed)[:,:2]
ax.scatter(y_stimmed_reduced[:,0], y_stimmed_reduced[:,1])
ax.axis('equal')
fig.savefig('/home/jgould/Downloads/nonrotational_dynamics_sjpca_discovered_space.svg')